Full name: Elsa Ingrid Daniela Erkfeldt
Civic registration number: 200309021228
LLM:

In [ ]:
import tensorflow as tf
import os
import torch.nn as nn
import numpy as np
import torch

if torch.cuda.is_available():
  device = torch.device("cuda")
else:
  device = torch.device("cpu")

Load the data from the PTB data set

In [ ]:
url = "https://raw.githubusercontent.com/wojzaremba/lstm/master/data/"
files = ["ptb.train.txt", "ptb.valid.txt", "ptb.test.txt"]

for f in files: 
    tf.keras.utils.get_file(f, url + f)

def load_data(filename):
    path = os.path.join(os.path.expanduser('~'), '.keras/datasets', filename)
    with open(path, 'r', encoding = 'utf-8') as f:
        return f.read().replace('\n', '<eos>')

train_text = load_data('ptb.train.txt')
valid_text = load_data('ptb.valid.txt')
test_text = load_data('ptb.test.txt')


In [ ]:
#Split the data into tokens and sort them
tokens = train_text.split()
vocab = sorted(list(set(tokens)))

valid_tok = valid_text.split()
valid_vocab = sorted(list(set(valid_tok)))

#Create mappings to be able to get an index from a word and vice versa
word2idx = {word: i for i, word in enumerate(vocab)}
idx2word = {i: word for i, word in enumerate(vocab)}

vocab_size = 10000
seq_length = 38
step = 1

#Create datasets with ser_length words as X and the word after that as y
def create_sequences(data, seq_length):
    X, y = [], []
    for i in range(0, len(data) - seq_length):
        X.append(data[i:i + seq_length])
        y.append(data[i + seq_length])
    return np.array(X), np.array(y)

X_train, y_train = create_sequences(tokens, seq_length)
X_valid, y_valid = create_sequences(valid_tok, seq_length)

In [ ]:
#Reuse the LSTM model from assignment 1
class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, num_layers):
        super(LSTMModel, self).__init__()
        self.embed = nn.Embedding(vocab_size, embed_size) #converts word IDs to vectors. vocab_size = num of distinct tokens. Embed_size = how big each word vector is 
        self.lstm = nn.LSTM(embed_size, hidden_size, num_layers, batch_first=True)
        self.linear = nn.Linear(hidden_size, vocab_size)
    def forward(self, x, h): #x:(batch_size, seq_length), h: (h0, c0), h0: zeros in shape (num_layers, batch_size, hidden_size), c0: same
        x = self.embed(x)
        out, (h,c) = self.lstm(x, h)
        out = out.reshape(out.size(0)*out.size(1), out.size(2))
        out = self.linear(out)
        return out, (h, c)

embed_size = 128
hidden_size = 5
num_layers = 2
learning_rate = 0.001

#Train the model on the data set we created
model = LSTMModel(vocab_size, embed_size, hidden_size, num_layers).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = learning_rate)

In [ ]:
"""#Validation
model.eval()

#predict
h0_predict = torch.zeros(num_layers, 1, hidden_size, , device=device)
c0_predict = torch.zeros(num_layers, 1, hidden_size, device = device)
val_per_time_test, _ =model(X_valid[0], (h0_predict, c0_predict))
print(val_per_time_test[-1])
id_for_last = torch.argmax(val_per_time_test[-1])
word_for_last = idx2word[id_for_last]
print(word_for_last)"""

aer


KeyError: 'I'

In [ ]:
"""#A function to calcualte the probability that the model will predict correctly

def predict_next(model, word, sentence, word2idx):


    model.eval()


    h0_predict = torch.zeros(num_layers, 1, hidden_size, , device=device)
    c0_predict = torch.zeros(num_layers, 1, hidden_size, device = device)
    val_per_time_test, _ = model(sentence, (h0_predict, c0_predict))
    print(val_per_time_test[-1])
    probs_next_word = val_per_time_test[-1]

    correct_next_word = word
    correct_next_word_id = word2idx[correct_next_word]
    prob_correct = probs_next_word[correct_next_word_id]


    id_for_last = torch.argmax(probs_next_word)
    word_for_last = idx2word[id_for_last]
    print(word_for_last)

    return prob_correct




predict_next(model, y_train[0], X_train[0], word2idx)

#Function to calculate perplexity
def calculate_perplexity(words):
    # PP = (P (tn | t1,..., t_n-1) ... P(t_3|t_1, t_2)P(t_2|t_1)P(t_1))^{-1/n}
    n = len(words)
    p_inside = 1
    for i in range(len(words), 0, -1):

        p_inside *= predict_next(model, words[i], words[0:i],word2idx)
    PP = (p_inside)**(-1/n)"""

In [ ]:
X_train_tensor = torch.tensor([[word2idx[w] for w in sentence] for sentence in X_train]).to(device)
y_train_tensor = torch.tensor([word2idx[w] for w in y_train]).to(device)

X_valid_tensor = torch.tensor([[word2idx[w] for w in sentence] for sentence in X_valid]).to(device)
y_valid_tensor = torch.tensor([word2idx[w] for w in y_valid]).to(device)

num_epochs = 10

#Training using a smoothed trigram
best_val_perplexity = np.inf
epochs_without_improvement = 0
patience = 3 

for epoch in range(num_epochs):
    model.train() 
    total_loss = 0
    num_batch = 0
    for batch_idx in range(0, len(X_train_tensor), seq_length):
        X_batch = X_train_tensor[batch_idx:batch_idx+seq_length]
        y_batch = y_train_tensor[batch_idx:batch_idx+seq_length]

        h0 = torch.zeros(num_layers, X_batch.size(0), hidden_size, device=device)
        c0 = torch.zeros(num_layers, y_batch.size(0), hidden_size, device=device)

        pred, _ = model(X_batch, (h0,c0))
        pred_reshaped = pred.reshape(X_batch.size(0), X_batch.size(1), -1)
        pred_last = pred_reshaped[:, -1, :]
        y_flattened = y_batch.view(-1)
        loss = criterion(pred_last, y_flattened)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        num_batch += 1
    model.eval()
    val_total_loss = 0
    val_num_batch = 0
    with torch.no_grad():
        for val_batch_idx in range(0, len(X_valid_tensor), seq_length):
            x_val_batch = X_valid_tensor[val_batch_idx:val_batch_idx+seq_length]
            y_val_batch = y_valid_tensor[val_batch_idx: val_batch_idx+ seq_length]
        
            h0_val = torch.zeros(num_layers, x_val_batch.size(0), hidden_size, device=device)
            c0_val = torch.zeros(num_layers, x_val_batch.size(0), hidden_size, device=device)

            predicted_val, _ = model(x_val_batch, (h0_val, c0_val))

            predicted_val_reshaped = predicted_val.reshape(x_val_batch.size(0), x_val_batch.size(1), -1)
#y_valid_flat = predicted_val.reshape(x_val_batch.size(0), x_val_batch.size(1), -1)
            pred_last_val = predicted_val_reshaped[:, -1, :]

            y_valid_flat = y_val_batch.view(-1)
            
        
            #_, pred_val_val = torch.max(predicted_val, 1)
            val_loss = criterion(pred_last_val, y_valid_flat)
            val_total_loss += val_loss.item()
            val_num_batch += 1             
    avg_train_loss = total_loss / num_batch
    avg_val_loss = val_total_loss / val_num_batch
    val_perplexity = torch.exp(torch.tensor(avg_val_loss)).item()
    print(f"Epoch {epoch}, loss = {avg_train_loss:.4f}, val_perplexity = {val_perplexity:.2f}")
    if val_perplexity < best_val_perplexity:
        best_val_perplexity = val_perplexity
        epochs_without_improvement = 0
        print("New best")
    else:
        epochs_without_improvement +=1
    if epochs_without_improvement >= patience:
        print("No improvement")
        break


print(f"best is {best_val_perplexity}")

Implement a smoothed triagram model
Then optimise the two smoothing parameters by traial and error to get the lowest validation perplexity

In [ ]:
def predict_next(model, word, sentence, word2idx, c_vals):
    model.eval()
    h0_predict = torch.zeros(num_layers, 1, hidden_size, device=device)
    c0_predict = torch.zeros(num_layers, 1, hidden_size, device = device)
    indices = [int(word2idx[w]) for w in sentence]
    tensor_indx_x = torch.tensor(indices, dtype=torch.long, device=device).unsqueeze(0)
    val_per_time_test, _ = model (tensor_indx_x, (h0_predict, c0_predict))
    print(val_per_time_test[-1])
    probs_next_word = torch.softmax(val_per_time_test[-1], dim=-1)
    correct_next_word = word
    correct_next_word_id = word2idx[correct_next_word]
    print(val_per_time_test)
    prob_correct = probs_next_word[correct_next_word_id]

    return float(prob_correct.item())
print(predict_next(model, y_train[0], X_train[0], word2idx))

def calculate_perplexity(word, sentence):
    # PP = (P (tn | t1,..., t_n-1) ... P(t_3|t_1, t_2)P(t_2|t_1)P(t_1))^{-1/n}
    n = len(words)
    p_inside = 1
    for i in range(1, len(words)-1):
        p_inside *= predict_next(model, words[i], words[:i],word2idx, c_vals)
    PP = (p_inside)**(-1/n)
print(f"x_train[0] is ", X_train[0], X_train[1], X_train[2])
print(f"y_train[0] is {y_train[0]} and {y_train[1]}, {y_train[2]}")

